# Step 1 — Binance 15분봉 데이터 & EMA200 차트

이 노트북은 Step 1 범위에 맞춰 Binance에서 BTC/ETH 15분봉 데이터를 불러오고, 
간단한 head/tail 확인과 EMA200이 포함된 차트를 출력합니다.


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(__file__).resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from data.binance_client import fetch_btc_eth_15m

plt.style.use("seaborn-v0_8")


In [ ]:
try:
    market_data = fetch_btc_eth_15m()
except RuntimeError as exc:
    print(f"실시간 데이터 수집 실패: {exc}\n랜덤 데이터를 이용해 예시를 진행합니다.")
    idx = pd.date_range(end=pd.Timestamp.utcnow(), periods=500, freq="15min")
    base = 20000 + np.cumsum(np.random.normal(0, 50, size=len(idx)))
    btc = pd.DataFrame(
        {
            "open": base,
            "high": base + np.random.uniform(10, 40, size=len(idx)),
            "low": base - np.random.uniform(10, 40, size=len(idx)),
            "close": base + np.random.uniform(-25, 25, size=len(idx)),
            "volume": np.random.uniform(100, 500, size=len(idx)),
        },
        index=idx,
    )
    eth = btc * 0.07
    market_data = {"BTCUSDT": btc, "ETHUSDT": eth}

for symbol, frame in market_data.items():
    display(symbol)
    display(frame.head())
    display(frame.tail())


In [ ]:
btc = market_data["BTCUSDT"].copy()
ema200 = btc["close"].ewm(span=200, adjust=False).mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(btc.index, btc["close"], label="Close", color="steelblue")
ax.plot(btc.index, ema200, label="EMA200", color="orange")
ax.set_title("BTCUSDT 15m Close & EMA200")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Price")
ax.legend()
plt.tight_layout()
plt.show()
